# SageMaker Serverless Inference


---

This notebook's CI test result for us-west-2 is as follows. CI test results in other regions can be found at the end of the notebook. 

![This us-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-2/inference|sm-serverless_inference_huggingface_text_classification|sm-serverless_inference_huggingface_text_classification.ipynb)

---

## HuggingFace Text Classification example

Amazon SageMaker Serverless Inference is a purpose-built inference option that makes it easy for you to deploy and scale ML models. Serverless Inference is ideal for workloads which have idle periods between traffic spurts and can tolerate cold starts. Serverless endpoints automatically launch compute resources and scale them in and out depending on traffic, eliminating the need to choose instance types or manage scaling policies. This takes away the undifferentiated heavy lifting of managing servers and helps developers focus on machine learning code.

This notebook uses **SageMaker Python SDK v3**:
- `ModelTrainer` to fine-tune a Hugging Face DistilBERT classifier on the [20 Newsgroups](http://qwone.com/~jason/20Newsgroups/) dataset
- `ModelBuilder` with `ServerlessInferenceConfig` to deploy the trained model to a serverless endpoint

### Runtime

This notebook takes approximately 30 minutes to run.

### Contents

1. Setup
2. Data preprocessing
3. Model training with `ModelTrainer`
4. Serverless deployment with `ModelBuilder`
5. Endpoint invocation
6. Cleanup

## Table of Contents
- Setup
- Data Preprocessing
- Model Training
- Deployment
    - Endpoint Configuration (Adjust for Serverless)
    - Serverless Endpoint Creation
    - Endpoint Invocation
- Cleanup
- Conclusion

# Development Environment and Permissions 

## Setup

If you run this notebook in SageMaker Studio, you need to make sure `ipywidgets` is installed and restart the kernel, so please uncomment the code in the next cell, and run it.

In [ ]:
%%capture
import IPython
import sys

!{sys.executable} -m pip install ipywidgets
# IPython.Application.instance().kernel.do_shutdown(True)  # has to restart kernel so changes are used

Let's install the required packages from HuggingFace and SageMaker

In [ ]:
import sys

!{sys.executable} -m pip install -U pip
!{sys.executable} -m pip install "sagemaker>=3.0" "transformers>=4.41,<4.50" "datasets>=2.19,<3" "nltk>=3.8" "evaluate>=0.4.2" "scikit-learn>=1.4" "s3fs>=2023.1.0"

Make sure the SageMaker Python SDK version is >= 3.0.

In [ ]:
import sagemaker
import boto3

print(sagemaker.__version__)
assert tuple(int(x) for x in sagemaker.__version__.split(".")[:2]) >= (3, 0), (
    "This notebook requires sagemaker>=3.0"
)

In [ ]:
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
role = get_execution_role()
region = sess.boto_region_name
bucket = sess.default_bucket()
default_bucket_prefix = sess.default_bucket_prefix

s3_prefix = "huggingface_serverless/20_newsgroups"
if default_bucket_prefix:
    s3_prefix = f"{default_bucket_prefix}/{s3_prefix}"

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {bucket}")
print(f"sagemaker session region: {region}")

### Data Preparation

Now we'll download a dataset from the web on which we want to train the text classification model.

In this example, let us train the text classification model on the [`20 newsgroups dataset`](http://qwone.com/~jason/20Newsgroups/). The `20 newsgroups dataset` consists of 20000 messages taken from 20 Usenet newsgroups.

In [ ]:
import os
import shutil

data_dir = "20_newsgroups_bulk"
if os.path.exists(data_dir):  # cleanup existing data folder
    shutil.rmtree(data_dir)

In [ ]:
s3 = boto3.client("s3")
s3.download_file(
    f"sagemaker-example-files-prod-{region}",
    "datasets/text/20_newsgroups/20_newsgroups_bulk.tar.gz",
    "20_newsgroups_bulk.tar.gz",
)

In [ ]:
!tar xzf 20_newsgroups_bulk.tar.gz
!ls 20_newsgroups_bulk

In [ ]:
from pathlib import Path

file_list = [os.path.join(data_dir, f) for f in os.listdir(data_dir)]
print("Number of files:", len(file_list))

In [ ]:
import pandas as pd

documents_count = 0
for file in file_list:
    df = pd.read_csv(file, header=None, names=["text"])
    documents_count = documents_count + df.shape[0]
print("Number of documents:", documents_count)

Let's inspect the dataset files and analyze the categories.

In [ ]:
categories_list = [Path(f).name for f in file_list]
categories_list

We can see that the dataset consists of 20 topics, each in different file.

Let us inspect the dataset to get some understanding about how the data and the label is provided in the dataset. 

In [ ]:
df = pd.read_csv("./20_newsgroups_bulk/rec.motorcycles", header=None, names=["text"])
df

In [ ]:
df["text"][0]

In [ ]:
df = pd.read_csv("./20_newsgroups_bulk/comp.sys.mac.hardware", header=None, names=["text"])
df

In [ ]:
df["text"][0]

As we can see from the above, there is a single file for each class in the dataset. Each record is just a plain text paragraphs with header, body, footer and quotes. We will need to process them into a suitable data format.

## Data Preprocessing
We need to preprocess the dataset to remove the header, footer, quotes, leading/trailing whitespace, extra spaces, tabs, and HTML tags/markups. 

Download the `nltk` tokenizer and other libraries

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
import re
import string

nltk.download("punkt")

In [ ]:
from sklearn.datasets._twenty_newsgroups import (
    strip_newsgroup_header,
    strip_newsgroup_quoting,
    strip_newsgroup_footer,
)

This following function will remove the header, footer and quotes (of earlier messages in each text).

In [ ]:
def strip_newsgroup_item(item):
    item = strip_newsgroup_header(item)
    item = strip_newsgroup_quoting(item)
    item = strip_newsgroup_footer(item)
    return item

The following function will take care of removing leading/trailing whitespace, extra spaces, tabs, and HTML tags/markups.

In [ ]:
def process_text(texts):
    final_text_list = []
    for text in texts:
        if not isinstance(text, str):
            text = ""
        text = text.lower().strip()
        text = re.sub(r"\[.*?\]", "", text)
        text = re.sub(r"https?://\S+|www\.\S+", "", text)
        text = re.sub("<.*?>+", "", text)
        text = re.sub("[%s]" % re.escape(string.punctuation), "", text)
        text = re.sub("\n", " ", text)
        text = re.sub(r"\w*\d\w*", "", text)
        text = re.sub(r"\s+", " ", text).strip()
        tokens = [w for w in word_tokenize(text) if w.isalpha()]
        final_text_list.append(" ".join(tokens))
    return final_text_list

Now we will read each of the `20_newsgroups` dataset files, call `strip_newsgroup_item` and `process_text` functions we defined earlier, and then aggregate all data into one dataframe.

In [ ]:
frames = []
for file in file_list:
    print(f"Processing {file}")
    label = Path(file).name
    df = pd.read_csv(file, header=None, names=["text"])
    df["text"] = df["text"].map(strip_newsgroup_item)
    df["text"] = process_text(df["text"].tolist())
    df["label"] = label
    frames.append(df)

all_categories_df = pd.concat(frames, ignore_index=True)

Let's inspect how many categories there are in our dataset.

In [ ]:
all_categories_df["label"].value_counts()

In our dataset there are 20 categories which is too much, so we will combine the sub-categories.

In [ ]:
# replace to politics
all_categories_df["label"].replace(
    {
        "talk.politics.misc": "politics",
        "talk.politics.guns": "politics",
        "talk.politics.mideast": "politics",
    },
    inplace=True,
)

# replace to recreational
all_categories_df["label"].replace(
    {
        "rec.sport.hockey": "recreational",
        "rec.sport.baseball": "recreational",
        "rec.autos": "recreational",
        "rec.motorcycles": "recreational",
    },
    inplace=True,
)

# replace to religion
all_categories_df["label"].replace(
    {
        "soc.religion.christian": "religion",
        "talk.religion.misc": "religion",
        "alt.atheism": "religion",
    },
    inplace=True,
)

# replace to computer
all_categories_df["label"].replace(
    {
        "comp.windows.x": "computer",
        "comp.sys.ibm.pc.hardware": "computer",
        "comp.os.ms-windows.misc": "computer",
        "comp.graphics": "computer",
        "comp.sys.mac.hardware": "computer",
    },
    inplace=True,
)
# replace to sales
all_categories_df["label"].replace({"misc.forsale": "sales"}, inplace=True)

# replace to science
all_categories_df["label"].replace(
    {
        "sci.crypt": "science",
        "sci.electronics": "science",
        "sci.med": "science",
        "sci.space": "science",
    },
    inplace=True,
)

Now we are left with 6 categories, which is much better.

In [ ]:
all_categories_df["label"].value_counts()

Let's calculate number of words for each row.

In [ ]:
all_categories_df["word_count"] = all_categories_df["text"].apply(lambda x: len(str(x).split()))
all_categories_df.head()

Let's get basic statistics about the dataset.

In [ ]:
all_categories_df["word_count"].describe()

We can see that the mean value is around 159 words. However, there are outliers, such as a text with 11351 words. This can make it harder for the model to result in good performance. We will take care to drop those rows.

Let's drop empty rows first.

In [ ]:
no_text = all_categories_df[all_categories_df["word_count"] == 0]
print(len(no_text))

# drop these rows
all_categories_df.drop(no_text.index, inplace=True)

Let's drop the rows that are longer than 256 words, as it is a length close to the mean value of the word count. This is done to make it easy for the model to train without outliers. 

In [ ]:
long_text = all_categories_df[all_categories_df["word_count"] > 256]
print(len(long_text))

# drop these rows
all_categories_df.drop(long_text.index, inplace=True)

In [ ]:
all_categories_df["label"].value_counts()

Let's get basic statistics about the dataset after our outliers fixes.

In [ ]:
all_categories_df["word_count"].describe()

This looks much more balanced.

Now we drop the `word_count` columns as we will not need it anymore.

In [ ]:
all_categories_df.drop(columns="word_count", axis=1, inplace=True)

In [ ]:
all_categories_df

Let's convert categorical label to integer number, in order to prepare the dataset for training.

In [ ]:
categories = all_categories_df["label"].unique().tolist()
categories

In [ ]:
categories.index("recreational")

In [ ]:
all_categories_df["label"] = all_categories_df["label"].apply(lambda x: categories.index(x))

In [ ]:
all_categories_df["label"].value_counts()

We partition the dataset into 80% training and 20% validation set and save to `csv` files.

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(all_categories_df, test_size=0.2)

In [ ]:
train_df.to_csv("train.csv", index=None)

In [ ]:
test_df.to_csv("test.csv", index=None)

Let's inspect the label distribution in the training dataset

In [ ]:
train_df["label"].value_counts()

Let's inspect the label distribution in the test dataset

In [ ]:
test_df["label"].value_counts()

## Tokenization 

A tokenizer is in charge of preparing the inputs for a model. The library contains tokenizers for all the models. Most of the tokenizers are available in two flavors: a full python implementation and a “Fast” implementation based on the Rust library [tokenizers](https://github.com/huggingface/tokenizers). The “Fast” implementations allows:

 - A significant speed-up in particular when doing batched tokenization.
 - Additional methods to map between the original string (character and words) and the token space (e.g. getting the index of the token comprising a given character or the span of characters corresponding to a given token). 

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

# tokenizer used in preprocessing
tokenizer_name = "distilbert-base-uncased"

In [ ]:
# download tokenizer
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

### Load train and test datasets

Let's create a [Dataset](https://huggingface.co/docs/datasets/loading_datasets.html) from our local `csv` files for training and test we saved earlier.

In [ ]:
dataset = load_dataset("csv", data_files={"train": "train.csv", "test": "test.csv"})

In [ ]:
dataset

In [ ]:
dataset["train"]

In [ ]:
dataset["train"][0]

In [ ]:
dataset["test"]

In [ ]:
dataset["test"][0]

In [ ]:
# tokenizer helper function
def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

In [ ]:
train_dataset = dataset["train"]
test_dataset = dataset["test"]

### Tokenize train and test datasets

Let's tokenize the train dataset

In [ ]:
train_dataset = train_dataset.map(tokenize, batched=True)

Let's tokenize the test dataset

In [ ]:
test_dataset = test_dataset.map(tokenize, batched=True)

### Set format for PyTorch

In [ ]:
train_dataset = train_dataset.rename_column("label", "labels")
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset = test_dataset.rename_column("label", "labels")
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

## Uploading data to `sagemaker_session_bucket`

After we processed the datasets, we are going to upload it to S3.

In [ ]:
training_input_path = f"s3://{bucket}/{s3_prefix}/train"
test_input_path = f"s3://{bucket}/{s3_prefix}/test"

train_dataset.save_to_disk(training_input_path)
test_dataset.save_to_disk(test_input_path)

print(training_input_path)
print(test_input_path)

## Training the Hugging Face model for supervised text classification

SageMaker Python SDK v3 replaces framework Estimators with `ModelTrainer`. You provide:
- a training image from `image_uris.retrieve(...)`
- `SourceCode` pointing at `train.py`
- `Compute` for the instance shape
- hyperparameters that become CLI flags inside the container

SageMaker launches the Hugging Face / PyTorch container, syncs `./code` and channel data under `/opt/ml/input/data`, then runs:

```bash
python train.py --epochs 1 --model_name distilbert-base-uncased --num_labels 6
```

Useful container environment variables:
- `SM_MODEL_DIR`: where model artifacts are written (uploaded to Amazon S3 after training)
- `SM_NUM_GPUS`: GPU count on the host
- `SM_CHANNEL_TRAIN` / `SM_CHANNEL_TEST`: channel mount paths

We create a metric_definition dictionary that contains regex-based definitions that will be used to parse the job logs and extract metrics

In [ ]:
from sagemaker.train.configs import MetricDefinition

metric_definitions = [
    MetricDefinition(name="loss", regex="'loss': ([0-9]+(.|e\-)[0-9]+),?"),
    MetricDefinition(name="learning_rate", regex="'learning_rate': ([0-9]+(.|e\-)[0-9]+),?"),
    MetricDefinition(name="eval_loss", regex="'eval_loss': ([0-9]+(.|e\-)[0-9]+),?"),
    MetricDefinition(name="eval_accuracy", regex="'eval_accuracy': ([0-9]+(.|e\-)[0-9]+),?"),
    MetricDefinition(name="eval_f1", regex="'eval_f1': ([0-9]+(.|e\-)[0-9]+),?"),
    MetricDefinition(name="eval_precision", regex="'eval_precision': ([0-9]+(.|e\-)[0-9]+),?"),
    MetricDefinition(name="eval_recall", regex="'eval_recall': ([0-9]+(.|e\-)[0-9]+),?"),
    MetricDefinition(name="eval_runtime", regex="'eval_runtime': ([0-9]+(.|e\-)[0-9]+),?"),
    MetricDefinition(name="eval_samples_per_second", regex="'eval_samples_per_second': ([0-9]+(.|e\-)[0-9]+),?"),
    MetricDefinition(name="epoch", regex="'epoch': ([0-9]+(.|e\-)[0-9]+),?"),
]

## Creating a ModelTrainer and starting a training job

In [ ]:
from sagemaker.core import image_uris
from sagemaker.core.shapes import StoppingCondition
from sagemaker.core.training.configs import InputData
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import Compute, SourceCode

hyperparameters = {
    "epochs": 1,
    "model_name": "distilbert-base-uncased",
    "num_labels": 6,
}

train_instance_type = "ml.g5.2xlarge"
training_image = image_uris.retrieve(
    framework="huggingface",
    region=region,
    version="4.48.0",
    py_version="py311",
    instance_type=train_instance_type,
    image_scope="training",
    base_framework_version="pytorch2.3.0",
)
print(training_image)

Define a SageMaker `ModelTrainer` with resource configuration and hyperparameters to train text classification on the 20 Newsgroups dataset.

In [ ]:
model_trainer = ModelTrainer(
    training_image=training_image,
    role=role,
    source_code=SourceCode(
        source_dir="./code",
        entry_script="train.py",
        requirements="requirements.txt",
    ),
    compute=Compute(
        instance_type=train_instance_type,
        instance_count=1,
        volume_size_in_gb=256,
    ),
    hyperparameters=hyperparameters,
    stopping_condition=StoppingCondition(max_runtime_in_seconds=3600),
    base_job_name="hf-serverless-20news",
).with_metric_definitions(metric_definitions)

In [ ]:
model_trainer.train(
    input_data_config=[
        InputData(channel_name="train", data_source=training_input_path),
        InputData(channel_name="test", data_source=test_input_path),
    ]
)

## Deployment

### Serverless configuration

#### Memory size - `memory_size_in_mb`
Your serverless endpoint has a minimum RAM size of **1024 MB (1 GB)**, and the maximum RAM size you can choose is 6144 MB (6 GB). Valid sizes are **1024**, **2048**, **3072**, **4096**, **5120**, or **6144** MB. Serverless Inference auto-assigns compute resources proportional to the memory you select.

#### Concurrent invocations - `max_concurrency`
This is the maximum number of concurrent invocations your serverless endpoint can process at any given time. Range: 1–200.

In SDK v3, pass `ServerlessInferenceConfig` to `ModelBuilder.deploy(...)` via `inference_config`.

In [ ]:
from sagemaker.core.inference_config import ServerlessInferenceConfig

serverless_config = ServerlessInferenceConfig(
    memory_size_in_mb=6144,
    max_concurrency=1,
)

### Serverless endpoint creation
Build a deployable model from the `ModelTrainer` result, then deploy it with the serverless configuration.

In [ ]:
%%time
import json
import uuid
from sagemaker.serve import ModelBuilder

inference_image = image_uris.retrieve(
    framework="huggingface",
    region=region,
    version="4.48.0",
    py_version="py311",
    instance_type="ml.m5.xlarge",
    image_scope="inference",
    base_framework_version="pytorch2.3.0",
)

endpoint_name = f"hf-serverless-20news-{str(uuid.uuid4())[:8]}"
model_name = f"hf-serverless-20news-model-{str(uuid.uuid4())[:8]}"

model_builder = ModelBuilder(
    model=model_trainer,
    image_uri=inference_image,
    role_arn=role,
    sagemaker_session=sess,
)
model_builder.build(model_name=model_name)
endpoint = model_builder.deploy(
    endpoint_name=endpoint_name,
    inference_config=serverless_config,
)
print(endpoint_name)

## Endpoint Invocation

Using few samples, you can now invoke the SageMaker endpoint to get predictions.

In [ ]:
def predict_sentence(sentence: str) -> None:
    raw = endpoint.invoke(
        body=json.dumps({"inputs": sentence}),
        content_type="application/json",
    )
    payload = raw if isinstance(raw, (bytes, bytearray)) else getattr(raw, "body", raw)
    if hasattr(payload, "read"):
        payload = payload.read()
    if isinstance(payload, (bytes, bytearray)):
        payload = payload.decode("utf-8")
    result = json.loads(payload) if isinstance(payload, str) else payload
    index = int(result[0]["label"].split("LABEL_")[1])
    print(categories[index])

In [ ]:
sentences = [
    "The modem is an internal AT/(E)ISA 8-bit card (just a little longer than a half-card).",
    "In the cage I usually wave to bikers.  They usually don't wave back.  My wife thinks it's strange but I don't care.",
    "Voyager has the unusual luck to be on a stable trajectory out of the solar system.",
]

# using the same processing logic that we used during data preparation for training
processed_sentences = process_text(sentences)

for sentence in processed_sentences:
    predict_sentence(sentence)

## Clean up

Endpoints should be deleted when no longer in use, since (per the [SageMaker pricing page](https://aws.amazon.com/sagemaker/pricing/)) they're billed by time deployed.

In [ ]:
from sagemaker.core.resources import EndpointConfig, Model

endpoint.delete()
try:
    EndpointConfig.get(endpoint_name).delete()
except Exception as exc:
    print(f"EndpointConfig cleanup skipped: {exc}")
try:
    Model.get(model_name).delete()
except Exception as exc:
    print(f"Model cleanup skipped: {exc}")

## Conclusion

In this notebook you ran a SageMaker Training Job with SDK v3 `ModelTrainer` to fine-tune a Hugging Face transformer for text classification on the 20 Newsgroups dataset. You then deployed the model to a SageMaker Serverless Endpoint with `ModelBuilder` and `ServerlessInferenceConfig`, invoked it with sample sentences, and cleaned up the endpoint resources.

As next steps, try your own dataset, raise `max_concurrency`, or compare latency/cost against a real-time endpoint.

## Notebook CI Test Results

This notebook was tested in multiple regions. The test results are as follows, except for us-west-2 which is shown at the top of the notebook.

![This us-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-1/inference|sm-serverless_inference_huggingface_text_classification|sm-serverless_inference_huggingface_text_classification.ipynb)

![This us-east-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-2/inference|sm-serverless_inference_huggingface_text_classification|sm-serverless_inference_huggingface_text_classification.ipynb)

![This us-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-1/inference|sm-serverless_inference_huggingface_text_classification|sm-serverless_inference_huggingface_text_classification.ipynb)

![This ca-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ca-central-1/inference|sm-serverless_inference_huggingface_text_classification|sm-serverless_inference_huggingface_text_classification.ipynb)

![This sa-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/sa-east-1/inference|sm-serverless_inference_huggingface_text_classification|sm-serverless_inference_huggingface_text_classification.ipynb)

![This eu-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-1/inference|sm-serverless_inference_huggingface_text_classification|sm-serverless_inference_huggingface_text_classification.ipynb)

![This eu-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-2/inference|sm-serverless_inference_huggingface_text_classification|sm-serverless_inference_huggingface_text_classification.ipynb)

![This eu-west-3 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-3/inference|sm-serverless_inference_huggingface_text_classification|sm-serverless_inference_huggingface_text_classification.ipynb)

![This eu-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-central-1/inference|sm-serverless_inference_huggingface_text_classification|sm-serverless_inference_huggingface_text_classification.ipynb)

![This eu-north-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-north-1/inference|sm-serverless_inference_huggingface_text_classification|sm-serverless_inference_huggingface_text_classification.ipynb)

![This ap-southeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-1/inference|sm-serverless_inference_huggingface_text_classification|sm-serverless_inference_huggingface_text_classification.ipynb)

![This ap-southeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-2/inference|sm-serverless_inference_huggingface_text_classification|sm-serverless_inference_huggingface_text_classification.ipynb)

![This ap-northeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-1/inference|sm-serverless_inference_huggingface_text_classification|sm-serverless_inference_huggingface_text_classification.ipynb)

![This ap-northeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-2/inference|sm-serverless_inference_huggingface_text_classification|sm-serverless_inference_huggingface_text_classification.ipynb)

![This ap-south-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-south-1/inference|sm-serverless_inference_huggingface_text_classification|sm-serverless_inference_huggingface_text_classification.ipynb)
